# Random Forest — Clasificador

**Repositorio:** `autodidact/both/random_forest/`

---

## 1. Idea General e Intuición

Random Forest es un **método de ensamble** basado en **Bagging** (*Bootstrap Aggregating*). Construye múltiples Árboles de Decisión, cada uno entrenado sobre un subconjunto aleatorio de datos y features, y combina sus predicciones.

**Dos fuentes de aleatoriedad (doble desacoplamiento):**
1. **Bootstrap de muestras:** Cada árbol se entrena con una muestra aleatoria *con reemplazo* del training set (≈63% de las muestras originales).
2. **Selección aleatoria de features:** En cada nodo del árbol solo se evalúan $m$ features elegidas al azar (no todas las $d$).

**Clasificación:** cada árbol vota por una clase → se elige la clase más votada (mayoría).

**¿Por qué funciona?** La combinación de árboles *poco correlacionados* reduce la varianza sin aumentar el sesgo. Cada árbol es un estimador con alta varianza (sobreajusta), pero el promedio de muchos estimadores no correlacionados tiene varianza mucho menor:

$$\text{Var}(\bar{X}) = \frac{\sigma^2}{n} \quad \text{(si son independientes)}$$

En la práctica los árboles tienen correlación $\rho > 0$, por lo que:

$$\text{Var}(\bar{X}) = \rho\sigma^2 + \frac{1-\rho}{n}\sigma^2$$

La aleatoriedad en features reduce $\rho$, lo que reduce la varianza del ensamble.

## 2. Fundamento Matemático

### 2.1 Bootstrap Sampling

Para cada árbol $t = 1, \ldots, T$:
$$\mathcal{D}_t = \text{Bootstrap}(\mathcal{D}) = \{(x_i, y_i) \sim \mathcal{D} \text{ con reemplazo}\}_{i=1}^{n}$$

Los ejemplos no seleccionados (≈37%) se llaman **Out-of-Bag (OOB)** y sirven como conjunto de validación gratuito.

### 2.2 Criterio de División de Nodos

En clasificación, el criterio más común es la **Impureza de Gini**:

$$\text{Gini}(t) = 1 - \sum_{k=1}^{K} p_k^2$$

Donde $p_k$ es la proporción de la clase $k$ en el nodo $t$.

Alternativa: **Entropía de Shannon**:
$$H(t) = -\sum_{k=1}^{K} p_k \log_2 p_k$$

La ganancia de información al dividir el nodo $t$ con el umbral $s$:
$$\Delta I = I(t) - \frac{n_L}{n} I(t_L) - \frac{n_R}{n} I(t_R)$$

### 2.3 Predicción Final (Votación)

$$\hat{y} = \underset{k}{\text{argmax}} \sum_{t=1}^{T} \mathbb{1}[\hat{y}_t = k]$$

Donde $\hat{y}_t$ es la predicción del árbol $t$.

### 2.4 Importancia de Features (MDI)

La importancia de la feature $j$ por **Mean Decrease in Impurity**:

$$\text{Imp}(j) = \frac{1}{T} \sum_{t=1}^{T} \sum_{\substack{v \in \text{nodos de } t \\ \text{que usan } j}} \frac{n_v}{n} \Delta I(v)$$

Donde $n_v$ es el número de muestras que llegan al nodo $v$.

## 3. ¿Cuándo Usarlo y Cuándo NO?

### ✅ Úsalo cuando:
- Tienes un **dataset tabular** de tamaño pequeño a grande.
- Las features son de **tipos mixtos** (numéricas y categóricas).
- No quieres preocuparte mucho por el **preprocesamiento** (no requiere escalado).
- Necesitas un modelo **robusto a outliers** y valores faltantes.
- Quieres **importancia de features** interpretable.

### ❌ NO lo uses cuando:
- Necesitas un modelo **altamente interpretable** (un solo árbol es más legible).
- El dataset es **muy grande** y la memoria es limitada (guarda todos los árboles).
- Tus features son **de alta dimensión y dispersas** (texto sin reducción → XGBoost/LightGBM son mejores).
- Necesitas **predicción en tiempo real ultrarrápida** (latencia de inferencia alta).

## 4. Métricas de Evaluación

Las mismas que cualquier clasificador:

| Métrica | Cuándo usarla |
|---|---|
| **Accuracy** | Clases balanceadas |
| **F1-Score (macro/weighted)** | Clases desbalanceadas |
| **ROC-AUC** | Comparar poder discriminativo |
| **OOB Score** | Estimación de generalización *gratuita* sin CV |

**OOB Score:** Cada muestra de entrenamiento actúa como dato de test en todos los árboles que no la usaron. Esto da una estimación casi tan buena como una CV de 5-fold, sin coste computacional extra.

## 5. Problemas y Casos Raros

### 5.1 Sobreajuste con Árboles Muy Profundos
Aunque RF es más robusto que un árbol único, con `max_depth=None` y pocas muestras puede sobreajustar.  
**Solución:** Usar `max_depth`, `min_samples_leaf`, `min_samples_split`.

### 5.2 Sesgo en Importancia MDI (Mean Decrease Impurity)
MDI favorece features con **mayor número de categorías o valores únicos** (features continuas > categóricas binarias). Es un sesgo conocido.  
**Solución:** Usar `permutation_importance` de sklearn, que es agnóstica al tipo de feature.

### 5.3 Clases Desbalanceadas
El voto mayoritario amplifica el desequilibrio.  
**Solución:** `class_weight='balanced'` o `BalancedRandomForestClassifier` (imblearn).

### 5.4 Datos de Alta Dimensión y Dispersos
Con $d \gg n$, RF puede producir importancias ruidosas.  
**Solución:** Reducir features con selección previa o usar `max_features` pequeño.

### 5.5 Extrapolación
RF **no puede extrapolar** más allá del rango de entrenamiento (promedio de árboles acotado al rango visto). Crítico en series temporales.

## 6. Mejora de Resultados

| Parámetro | Efecto | Recomendación |
|---|---|---|
| `n_estimators` | ↑ → mejor rendimiento hasta saturar | 100-500; más no hace daño pero sí cuesta tiempo |
| `max_depth` | ↑ → más varianza, menos sesgo | `None` (completo) o buscar con CV |
| `max_features` | ↑ → más correlación entre árboles | `'sqrt'` para clasificación, `'log2'` alternativa |
| `min_samples_leaf` | ↑ → más suavizado | 1-5, aumentar si hay ruido |
| `class_weight` | `'balanced'` para desbalanceo | Usar siempre si clases desbalanceadas |
| `oob_score` | `True` → estimación de error gratuita | Activar siempre en desarrollo |

**Regla empírica para `max_features`:**
- Clasificación: $m = \sqrt{d}$
- Regresión: $m = d/3$

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
)
from sklearn.tree import plot_tree

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
SEED = 42
print('✅ Librerías cargadas.')

---
## 7. Clasificación — Breast Cancer

In [ ]:
# =============================================================================
# CARGA Y SPLIT
# =============================================================================
bc = load_breast_cancer(as_frame=True)
X, y = bc.data, bc.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Clases: {bc.target_names} | Distribución: {pd.Series(y).value_counts().to_dict()}')

In [ ]:
# =============================================================================
# MODELO BASE CON OOB SCORE
# oob_score=True activa la estimación out-of-bag sin necesidad de CV.
# Random Forest NO requiere escalado de features.
# =============================================================================
rf = RandomForestClassifier(
    n_estimators=200,
    max_features='sqrt',       # Selección aleatoria de sqrt(d) features por nodo
    max_depth=None,            # Árboles completos
    min_samples_leaf=1,
    class_weight='balanced',   # Compensa desbalanceo automáticamente
    oob_score=True,
    random_state=SEED,
    n_jobs=-1                  # Paralelizar en todos los núcleos
)
rf.fit(X_train, y_train)
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print(f'OOB Score  : {rf.oob_score_:.4f}  ← estimación gratuita sin CV')
print(f'Accuracy   : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC    : {roc_auc_score(y_test, y_proba):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=bc.target_names))

In [ ]:
# =============================================================================
# VISUALIZACIÓN 1: Curva de Convergencia — OOB Error vs n_estimators
# Muestra cuántos árboles son suficientes (punto de saturación)
# =============================================================================
oob_errors = []
n_tree_range = range(10, 301, 10)

for n in n_tree_range:
    rf_tmp = RandomForestClassifier(n_estimators=n, oob_score=True,
                                     max_features='sqrt', class_weight='balanced',
                                     random_state=SEED, n_jobs=-1)
    rf_tmp.fit(X_train, y_train)
    oob_errors.append(1 - rf_tmp.oob_score_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(n_tree_range), oob_errors, 'o-', color='steelblue', lw=2, ms=4)
ax.set_xlabel('Número de Árboles (n_estimators)', fontsize=12)
ax.set_ylabel('OOB Error Rate', fontsize=12)
ax.set_title('Curva de Convergencia: OOB Error vs n_estimators', fontsize=13, fontweight='bold')
ax.axhline(min(oob_errors), color='crimson', linestyle='--',
           label=f'Error mínimo: {min(oob_errors):.4f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VISUALIZACIÓN 2: Importancia de Features
# Comparamos MDI (incorporado) vs Permutation Importance (más robusto)
# =============================================================================
feat_names = bc.feature_names

# MDI (Mean Decrease in Impurity) — sesgo hacia features continuas con más valores
mdi_imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)

# Permutation Importance — más honesto, libre de sesgo MDI
perm_imp_res = permutation_importance(rf, X_test, y_test, n_repeats=20,
                                       random_state=SEED, n_jobs=-1)
perm_imp = pd.Series(perm_imp_res.importances_mean, index=feat_names).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# MDI
top_n = 12
mdi_top = mdi_imp.head(top_n)
axes[0].barh(mdi_top.index[::-1], mdi_top.values[::-1], color='steelblue', alpha=0.8)
axes[0].set_title('Importancia MDI (incorporada)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importancia media (Gini)')

# Permutation
perm_top = perm_imp.head(top_n)
axes[1].barh(perm_top.index[::-1], perm_top.values[::-1], color='seagreen', alpha=0.8)
axes[1].set_title('Permutation Importance (test set)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Reducción media de accuracy')

plt.suptitle('Importancia de Features: MDI vs Permutation\nBreast Cancer Dataset',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VISUALIZACIÓN 3: Matriz de Confusión + Curva ROC
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=bc.target_names).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Matriz de Confusión', fontsize=13, fontweight='bold')

RocCurveDisplay.from_estimator(rf, X_test, y_test, ax=axes[1], name='Random Forest')
axes[1].plot([0,1],[0,1],'k--', label='Azar')
axes[1].set_title('Curva ROC', fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle('Random Forest Clasificador — Breast Cancer',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VISUALIZACIÓN 4: Uno de los Árboles del Bosque (profundidad limitada)
# Ilustra qué aspecto tiene un árbol individual del ensamble
# =============================================================================
fig, ax = plt.subplots(figsize=(20, 6))
plot_tree(
    rf.estimators_[0],        # Primer árbol del bosque
    max_depth=3,              # Solo 3 niveles para legibilidad
    feature_names=bc.feature_names,
    class_names=bc.target_names,
    filled=True,
    rounded=True,
    ax=ax,
    fontsize=8
)
ax.set_title('Árbol #1 del Bosque (profundidad=3 para visualización)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# HYPERPARAMETER TUNING con GridSearchCV
# =============================================================================
param_grid = {
    'n_estimators':    [100, 200],
    'max_depth':       [None, 10, 20],
    'max_features':    ['sqrt', 'log2'],
    'min_samples_leaf':[1, 3]
}

grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', oob_score=False,
                           random_state=SEED, n_jobs=-1),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
)
grid.fit(X_train, y_train)

print(f'Mejores parámetros: {grid.best_params_}')
print(f'Mejor ROC-AUC (CV): {grid.best_score_:.4f}')

best_rf = grid.best_estimator_
y_pred_best  = best_rf.predict(X_test)
y_proba_best = best_rf.predict_proba(X_test)[:, 1]
print(f'ROC-AUC en Test  : {roc_auc_score(y_test, y_proba_best):.4f}')

---
## 8. Resumen del Clasificador

| Aspecto | Detalle |
|---|---|
| **Tipo** | Ensamble de árboles (Bagging) |
| **Preprocesamiento** | No requiere escalado |
| **Interpretabilidad** | Media (importancia de features) |
| **Robustez a outliers** | Alta |
| **Hiperparámetro clave** | `n_estimators`, `max_features`, `max_depth` |

### ✅ Checklist:
- [ ] `oob_score=True` activado durante desarrollo
- [ ] `class_weight='balanced'` si hay desbalanceo
- [ ] Comparar MDI vs Permutation Importance
- [ ] Verificar curva de convergencia antes de fijar `n_estimators`

---
*Ver notebook de regresión: `rf_regressor.ipynb`*